<p>
  <img src="../img/lcms-icon.png" height="60" alt="LCMS"/>
  &nbsp;&nbsp;
  <img src="../img/usfslogo.png" height="60" alt="USDA Forest Service"/>
  &nbsp;&nbsp;
  <img src="../img/rcr-logo-new.png" height="60" alt="RedCastle Resources"/>
</p>

# LCMS Introduction: What Can 40 Years of Land Cover Change Data Tell You?
### Using USFS Landscape Change Monitoring System (LCMS) in Google Earth Engine

Copyright 2026 Lila Leatherman

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

---

The **[Landscape Change Monitoring System (LCMS)](https://www.fs.usda.gov/lcms)** is an annual, 30 m resolution, wall-to-wall mapping product produced by the USDA Forest Service's Geospatial Technology and Applications Center (GTAC). It covers the contiguous United States and southeast Alaska from **1985 through the present** — one of the longest continuous land cover time series available at 30 m for North America.

Each year of LCMS produces **three thematic maps**, each answering a different question about the same landscape:

| Band | Question | Example classes |
|------|----------|-----------------| 
| `Land_Cover` | **What is on the ground?** | Trees, Shrubs, Grass/Forb/Herb, Barren, Water |
| `Land_Use` | **How is the land used?** | Forest, Agriculture, Developed, Rangeland |
| `Change` | **What changed, and how?** | Tree Removal, Wildfire, Successional Growth, Stable |

This notebook is a **hands-on orientation** to these three products. By the end you will have:
- Explored all three LCMS bands on an interactive map
- Charted how land cover composition has shifted over 40 years
- Identified where and when forest disturbance occurred in the Change record

---

**Prerequisites**
- A Google Earth Engine (GEE) account — [sign up here](https://earthengine.google.com/signup/)
- A GEE Cloud project with billing enabled — [create one here](https://console.cloud.google.com/projectcreate)
- Python >= 3.9 with `earthengine-api` and `geeViz` installed:
  ```bash
  pip install earthengine-api geeViz
  ```

> **Tip:** This notebook runs sequentially top-to-bottom. When you are ready to go deeper, open **`hj_andrews_harvest_recovery.ipynb`** or **`macdunn_harvest_recovery.ipynb`** for site-specific analyses that build on everything introduced here.

## 1 · Setup — Imports and Authentication

In [1]:
import os, ee
from IPython.display import display, HTML

# ── Authentication ─────────────────────────────────────────────────────────────
# Run ee.Authenticate() once per machine to store credentials locally.
# After that, comment it out — only ee.Initialize() is needed on subsequent runs.
#ee.Authenticate()

# ── Initialization ─────────────────────────────────────────────────────────────
# ee.Initialize MUST come before geeViz imports. Importing geeViz first triggers
# its internal auth proxy with cached credentials that may belong to a different
# project.
EE_PROJECT = 'rcr-gee' # <- replace with your GEE Cloud project ID
ee.Initialize(project=EE_PROJECT)

# ── In-process HTTP server (prevents stale-layer bug) ─────────────────────────
# Without this, geeViz spawns a background subprocess from the system Python,
# not the venv — the map viewer then serves stale JS from a previous run.
os.environ['GEEVIZ_EEAUTH_MODE'] = 'auto'

# ── geeViz imports ─────────────────────────────────────────────────────────────
import geeViz.getImagesLib as gil
import geeViz.geeView
from geeViz.outputLib import charts as cl

Map = gil.Map
Map.port = 8080
Map.project = EE_PROJECT
Map.clearMap()

test = ee.Image(1).getInfo()
print('Test: ', test)
print('Earth Engine initialized successfully.')

Test:  {'type': 'Image', 'bands': [{'id': 'constant', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 1, 'max': 1}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}]}
Earth Engine initialized successfully.


## 2 · Study Area — H.J. Andrews Experimental Forest

The [H.J. Andrews Experimental Forest](https://andrewsforest.oregonstate.edu/) is a 64 km² LTER site in the western Oregon Cascades with a well-documented history: decades of experimental and commercial timber harvest (1950s–1990s), followed by near-cessation of cutting after the **1994 Northwest Forest Plan**. This before/after contrast is clearly visible in all three LCMS bands and makes it an ideal orientation site.

The analysis uses a bounding box that encompasses the forest plus surrounding Willamette National Forest context. **Replace the coordinates with any aoi or polygon** to explore LCMS in a different region.

In [2]:
START_YEAR = 1985
END_YEAR   = 2025

# ── Study area bounding box ───────────────────────────────────────────────────
# H.J. Andrews Experimental Forest, western Oregon Cascades.
# Swap these coordinates for any location to explore LCMS elsewhere.
# If your study area is in Alaska, Hawaii, or Puerto Rico / the US Virgin Islands, 
# update the study_area to "AK", "HI", or "PRUSVI".
bbox_coords = [-122.32, 44.18, -122.09, 44.28]
aoi = ee.Geometry.BBox(bbox_coords[0], bbox_coords[1], bbox_coords[2], bbox_coords[3])
study_area = "CONUS"


print('Study date range :', START_YEAR, '-', END_YEAR)
print('Study area type  :', aoi.getInfo()['type'])
area_km2 = aoi.area(maxError=500).divide(1e6).getInfo()
print(f'Bounding box area: {area_km2:,.0f} km2  (H.J. Andrews ~64 km2; aoi includes surrounding context)')

Study date range : 1985 - 2025
Study area type  : Polygon
Bounding box area: 204 km2  (H.J. Andrews ~64 km2; aoi includes surrounding context)


## 3 · Load LCMS and Explore the Data

LCMS v2025-11 covers **1985–2025** (41 annual images). The code below:

1. Loads the `ImageCollection` filtered to the study area
2. Prints the year range and available bands
3. Prints the class lookup tables for all three thematic bands — Land_Cover, Land_Use, and Change — so you can have the value table as an easy reference. 

The class tables are provided in this repo as a .json metadata file. They are also included as properties of the LCMS ImageCollection, but there are some issues with the properties (that we're addressing!!) so we're using the hard-coded ones here.

In [3]:
import json, pandas as pd
from pathlib import Path

# The v2025-11 asset is the latest version of LCMS.
LCMS_ASSET_2025 = 'projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11'

# Filter to study area
lcms = (
    ee.ImageCollection(LCMS_ASSET_2025)
    .filter(ee.Filter.eq('study_area', study_area))
    .filterBounds(aoi)
)

# Get some metadata about the collection
n_images = lcms.size().getInfo()
years    = lcms.aggregate_array('year').distinct().sort().getInfo()
bands    = lcms.first().bandNames().getInfo()
thematic = ['Change', 'Land_Cover', 'Land_Use']
raw_prob  = [b for b in bands if 'Raw' in b]
properties = lcms.first().propertyNames().getInfo()

print(f'Images in collection : {n_images}')
print(f'Years                : {years[0]}-{years[-1]}')
print(f'Thematic bands       : {thematic}')
print(f'Raw probability bands: {len(raw_prob)} bands (model confidence behind each classification)')
print(f'Properties            : {properties}')

# Load class metadata from the local JSON file.
# The EE asset stores class names as a single comma-delimited string. geeViz
# (autoViz) and cl.summarize_and_chart both split on commas to parse names, so
# "Insect, Disease, or Drought Stress" fragments into three entries, shifting
# all subsequent name→value mappings by 2 positions. The JSON uses a proper
# list, which avoids this entirely.
meta = json.loads((Path('..') / 'data' / 'metadata' / 'lcms_metadata.json').read_text())

change_names   = meta['Change_class_names']
change_values  = meta['Change_class_values']
change_palette = meta['Change_class_palette']

land_cover_names   = meta['Land_Cover_class_names']
land_cover_values  = meta['Land_Cover_class_values']
land_cover_palette = meta['Land_Cover_class_palette']

land_use_names   = meta['Land_Use_class_names']
land_use_values  = meta['Land_Use_class_values']
land_use_palette = meta['Land_Use_class_palette']

# Write the corrected lists back onto each image so autoViz and
# cl.summarize_and_chart read them instead of the asset's comma-delimited strings.
lcms = lcms.map(lambda img: img.set({
    'Change_class_names':       change_names,
    'Change_class_values':      change_values,
    'Change_class_palette':     change_palette,
    'Land_Cover_class_names':   land_cover_names,
    'Land_Cover_class_values':  land_cover_values,
    'Land_Cover_class_palette': land_cover_palette,
    'Land_Use_class_names':     land_use_names,
    'Land_Use_class_values':    land_use_values,
    'Land_Use_class_palette':   land_use_palette,
}))
img_for_metadata = ee.Image(lcms.first())  # used later for .copyProperties()

print('\nChange classes:')
for val, name in zip(change_values, change_names):
    print(f'  {val:>3}  {name}')

print('\nLand Cover classes:')
for val, name in zip(land_cover_values, land_cover_names):
    print(f'  {val:>3}  {name}')

print('\nLand Use classes:')
for val, name in zip(land_use_values, land_use_names):
    print(f'  {val:>3}  {name}')


Images in collection : 41
Years                : 1985-2025
Thematic bands       : ['Change', 'Land_Cover', 'Land_Use']
Raw probability bands: 22 bands (model confidence behind each classification)
Properties            : ['Land_Cover_class_names', 'year', 'Land_Use_class_values', 'startYear', 'Land_Use_class_palette', 'system:id', 'Change_class_names', 'endYear', 'version', 'system:time_start', 'Change_class_palette', 'Change_class_values', 'Land_Use_class_names', 'Land_Cover_class_palette', 'system:footprint', 'Land_Cover_class_values', 'study_area', 'system:version', 'system:asset_size', 'system:index', 'system:bands', 'system:band_names']

Change classes:
    1  Wind
    2  Hurricane
    3  Snow or Ice Transition
    4  Desiccation
    5  Inundation
    6  Prescribed Fire
    7  Wildfire
    8  Mechanical Land Transformation
    9  Tree Removal
   10  Defoliation
   11  Southern Pine Beetle
   12  Insect, Disease, or Drought Stress
   13  Other Loss
   14  Vegetation Successional Gr

## 4 · Interactive Map — The Three LCMS Products

The map below shows six layers you can toggle with the layer panel on the left:

| Layer | Band | Coverage |
|-------|------|----------|
| **Land Cover 2025** | `Land_Cover` | Most recent year — what is on the ground today |
| **Land Use 2025** | `Land_Use` | Most recent year — how the land is classified by use |
| **Change Agent 2025** | `Change` | Most recent year — what, if anything, changed |
| **Most Common Change Agent 1985–2025** | `Change` | Modal (most frequent) class per pixel over 40 years |
| **Most Severe Change Agent 1985–2025** | `Change` | Most severe class per pixel over 40 years |
| **Ever Disturbed 1985–2025** | `Change` | Any pixel with Tree Removal OR Wildfire in any year |

**Tips:**
- Click any pixel to query its class value
- Draw a polygon with the draw tools and click **"Chart Selected Area"** to get a time series for any custom sub-region
- Toggle layers on and off to compare the three LCMS products side by side

In [4]:
Map.clearMap()

lcms_recent = lcms.filter(ee.Filter.eq('year', END_YEAR))

# Because the LCMS v2025-11 asset contains class metadata in the image properties, we can use 'autoViz' 
# to automatically apply the correct class names and colors for each thematic band.

# ── Land Cover — most recent year ─────────────────────────────────────────────
Map.addLayer(
    lcms_recent.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    f'Land Cover {END_YEAR}',
    True,
)

# ── Land Use — most recent year ────────────────────────────────────────────────
Map.addLayer(
    lcms_recent.select('Land_Use'),
    {'autoViz': True, 'canAreaChart': True},
    f'Land Use {END_YEAR}',
    False,
)

# ── Change Agent — most recent year ───────────────────────────────────────────
Map.addLayer(
    lcms_recent.select('Change'),
    {'autoViz': True, 'canAreaChart': True},
    f'Change Agent {END_YEAR}',
    False,
)

# -- Most Common Change Agent (1985–2025) ───────────────────────────────────────────────
lcms_most_common_change = (
    ee.Image(lcms.select('Change').mode())
    .copyProperties(img_for_metadata)
)
Map.addLayer(
    lcms_most_common_change,
    {'autoViz': True, 'canAreaChart': True},
    'Most Common Change Agent 1985-2025',
    True,
)

# ── Most severe Change Agent (1985–2025) ──────────────────────────────────────
# Change agent codes are nominally arranged from "most" to "least" severe by class value
# The .min() reducer will identify the lowest (most severe) class value for each pixel across all years.
# .min() drops image metadata (class names/colors); .copyProperties() restores
# it so geeViz autoViz can look up the correct class symbology.
lcms_most_severe_change = (
    ee.Image(lcms.select('Change').min())
    .copyProperties(img_for_metadata)
)
Map.addLayer(
    lcms_most_severe_change,
    {'autoViz': True, 'canAreaChart': True},
    'Most Severe Change Agent 1985-2025',
    True,
)

# ── Ever Disturbed — Tree Removal OR Wildfire, any year ───────────────────────
name_to_val  = dict(zip(change_names, change_values))
TREE_REMOVAL = name_to_val['Tree Removal']   # numeric class value for Tree Removal
WILDFIRE     = name_to_val['Wildfire']        # numeric class value for Wildfire

ever_disturbed = (
    lcms.select('Change')
    .map(lambda img: img.eq(TREE_REMOVAL).Or(img.eq(WILDFIRE)))
    .max()                # 1 if disturbed in any year, 0 otherwise
    .selfMask()           # mask 0s so undisturbed pixels are transparent
    .rename('ever_disturbed')
)
ever_disturbed = ever_disturbed.set({
    'ever_disturbed_class_values':  [1],
    'ever_disturbed_class_names':   ['Tree Removal or Wildfire (any year 1985-2025)'],
    'ever_disturbed_class_palette': ['d54309'],
})
Map.addLayer(ever_disturbed, {'autoViz': True}, 'Ever Disturbed 1985-2025', False)

# ── Study area outline ─────────────────────────────────────────────────────────
Map.addLayer(
    ee.Feature(aoi, {}),
    {'layerType': 'geeVector', 'strokeColor': 'ffff00', 'strokeWidth': 2,
     'fillColor': '00000000'},
    'Study Area',
    True,
)

Map.centerObject(aoi, 11)
Map.view()

Adding layer: Land Cover 2025
Adding layer: Land Use 2025
Adding layer: Change Agent 2025
Adding layer: Most Common Change Agent 1985-2025
Adding layer: Most Severe Change Agent 1985-2025
Adding layer: Ever Disturbed 1985-2025
Adding layer: Study Area
Starting webmap


[geeViz.eeAuth] EE initialized via proxy: http://127.0.0.1:8891/ee-api (tenant_header=X-geeViz-Creds)


geeViz server at http://localhost:8080/geeView/
Using eeCreds proxy at http://127.0.0.1:8891/ee-api (creds=ee-persistent)
geeView URL: http://localhost:8080/geeView/?v=1785361416475


## 5 · Time Lapse — 40 Years of Change

The slider below animates each LCMS band from **1985 to 2025**. Use it to scan through the full record year by year and watch the landscape change.

**Tips:**
- The time lapses take a while to load — be patient!
- Use the **year slider** at the top of the map to step through time
- Switch between the three layers to compare how land cover, land use, and change evolved in the same year
- Draw a polygon and click **"Chart Selected Area"** while the time lapse is active to plot a class-area time series for any sub-region


In [5]:
Map.clearMap()

# ── Land Cover time lapse (1985–2025) ─────────────────────────────────────────
Map.addTimeLapse(
    lcms.select('Land_Cover'),
    {'autoViz': True, 'canAreaChart': True},
    'Land Cover 1985-2025',
     False,
)

# ── Land Use time lapse (1985–2025) ───────────────────────────────────────────
Map.addTimeLapse(
    lcms.select('Land_Use'),
    {'autoViz': True, 'canAreaChart': True},
    'Land Use 1985-2025',
    False,
)

# ── Change Agent time lapse (1985–2025) ───────────────────────────────────────
Map.addTimeLapse(
    lcms.select('Change'),
    {'autoViz': True, 'canAreaChart': True},
    'Change Agent 1985-2025',
    True,
)

Map.addLayer(aoi, {'layerType': 'geeVector', 'strokeColor': 'ffff00', 'strokeWidth': 2,
                   'fillColor': '00000000'}, 'Study Area', True)
Map.centerObject(aoi, 11)
Map.view()


Adding layer: Land Cover 1985-2025
Adding layer: Land Use 1985-2025
Adding layer: Change Agent 1985-2025
Adding layer: Study Area
Starting webmap
Using eeCreds proxy at http://127.0.0.1:8891/ee-api (creds=ee-persistent)
geeView URL: http://localhost:8080/geeView/?v=1785361416685


## 6 · Change Agents Over Time

> **Note on class filtering:** LCMS defines **16 Change classes**. At the H.J. Andrews (a wet Pacific Northwest forest), most classes — Wind, Hurricane, Desiccation, Southern Pine Beetle, Insect/Disease, and others — register near-zero values here and would clutter the chart with flat lines near zero. The plot below shows only the **5 most ecologically meaningful classes** for this site and forest type:
>
> | Class | Ecological role at HJA |
> |-------|------------------------|
> | **Stable** | No change detected — baseline; dominates in old-growth and mature forest |
> | **Tree Removal** | Harvest or clearing — the primary disturbance at HJA before 1994 |
> | **Vegetation Successional Growth** | Active recovery following harvest; canopy establishment |
> | **Wildfire** | Fire disturbance — sharp increase toward the end of the time series |
> | **Other Loss** | All other vegetation loss that can't be definitively attributed to the loss classes above |
>
> **All 16 classes are still computed.** The full DataFrame is printed at the end of the cell so you can inspect every class. Swap in a drier, more fire-prone study area and classes like Wildfire, Wind, or Insect/Disease will become much more prominent.

**What to look for:**
- **Successional Growth rising** through the 1980s–2000s as previously harvested stands recovered
- **Wildfire spike** in the mid-2020s, corresponding to the increasingly severe wildfire seasons
- **Stable dominating throughout** — most of the old-growth forest was never disturbed during the LCMS record
- **Other Loss fluctuating** at low levels — minor vegetation loss not attributable to the other disturbance types


In [6]:
import plotly.graph_objects as go

# Get the Change Agent time series for the study area and plot it as a line chart.
change_result = cl.summarize_and_chart(
    lcms,
    geometry=aoi,
    band_names='Change',
    scale=30,
    area_format='Percentage',
    title='Change Agents - H.J. Andrews Experimental Forest 1985-2025',
    chart_type='line',
    stacked=False,
    date_format='YYYY',
    width=950,
    height=440,
)

# ── Filter to 5 ecologically meaningful classes for Pacific NW forest ─────────
# All 16 classes were computed above. Only these 5 are plotted below.
# See change_result['df'] (printed at the end of this cell) for the full data.
FOCUS_CLASSES = [
    'Tree Removal',
    'Wildfire',
    'Vegetation Successional Growth',
    'Stable',
    'Other Loss',
]
focus_cols = [
    c for c in change_result['df'].columns
    if any(cls in c for cls in FOCUS_CLASSES)
]
n_all = len(change_result['df'].columns)
print(f'Displaying {len(focus_cols)} of {n_all} Change classes.')
print(f'Columns plotted: {focus_cols}')

# Colors sourced from the LCMS Change palette (data/lcms_metadata.json)
CLASS_COLORS = {
    'Wildfire':                       '#d54309',
    'Tree Removal':                   '#afde1c',
    'Other Loss':                     '#c291d5',
    'Vegetation Successional Growth': '#00a398',
    'Stable':                         '#3d4551',
    
}

fig_change = go.Figure()
for col in focus_cols:
    class_name = next((cls for cls in FOCUS_CLASSES if cls in col), col)
    fig_change.add_trace(go.Scatter(
        x=change_result['df'].index,
        y=change_result['df'][col].values,
        name=class_name,
        mode='lines+markers',
        line=dict(color=CLASS_COLORS.get(class_name, '#888888'), width=2),
        marker=dict(size=3),
    ))

fig_change.update_layout(
    title='Change Agents - H.J. Andrews Experimental Forest 1985-2025 (5 key classes)',
    xaxis=dict(title='Year', tickmode='linear', dtick=5),
    yaxis=dict(title='% of study area'),
    legend=dict(orientation='h', yanchor='top', y=-0.20, xanchor='left', x=0),
    width=950, height=480,
    template='plotly_white',
)
fig_change.show()


Displaying 5 of 10 Change classes.
Columns plotted: ['Other Loss', 'Vegetation Successional Growth', 'Stable', 'Tree Removal', 'Wildfire']


## 7 · Land Cover Over Time

The chart below shows how the percentage of each Land Cover class has evolved inside the study area from 1985 to 2025. The classes are **stacked** — the total always sums to 100%.

**What to look for at the H.J. Andrews:**
- **Trees dominating** throughout — this is a Pacific Northwest old-growth and plantation conifer forest
- A modest **dip in Trees** in the mid-2020s in response to late-summer drought and wildfire 


In [7]:
lc_result = cl.summarize_and_chart(
    lcms,
    geometry=aoi,
    band_names='Land_Cover',
    scale=30,
    area_format='Percentage',
    title='Land Cover - H.J. Andrews Experimental Forest 1985-2025',
    chart_type='line',
    stacked=True,
    date_format='YYYY',
    width=950,
    height=440,
)

lc_result['chart'].show()

# ── Print Trees cover % as a quick numerical reference ────────────────────────
trees_col = next(
    (c for c in lc_result['df'].columns if 'Trees' in c and 'Tall' not in c),
    None,
)
if trees_col:
    print(f'\nAnnual "Trees" cover (%) — column label: "{trees_col}"\n')
    print(
        lc_result['df'][[trees_col]]
        .rename(columns={trees_col: 'Trees (%)'})
        .to_markdown()
    )


Annual "Trees" cover (%) — column label: "Trees"

|      |   Trees (%) |
|-----:|------------:|
| 1985 |       95.99 |
| 1986 |       96.17 |
| 1987 |       96.4  |
| 1988 |       96.55 |
| 1989 |       96.67 |
| 1990 |       96.78 |
| 1991 |       97.02 |
| 1992 |       97.18 |
| 1993 |       97.37 |
| 1994 |       97.57 |
| 1995 |       97.76 |
| 1996 |       97.78 |
| 1997 |       97.81 |
| 1998 |       97.95 |
| 1999 |       98.02 |
| 2000 |       98.14 |
| 2001 |       98.27 |
| 2002 |       98.33 |
| 2003 |       98.03 |
| 2004 |       98.03 |
| 2005 |       98.13 |
| 2006 |       98.2  |
| 2007 |       98.26 |
| 2008 |       98.31 |
| 2009 |       98.41 |
| 2010 |       98.49 |
| 2011 |       98.66 |
| 2012 |       98.73 |
| 2013 |       98.78 |
| 2014 |       98.79 |
| 2015 |       98.64 |
| 2016 |       98.51 |
| 2017 |       98.46 |
| 2018 |       98.49 |
| 2019 |       98.51 |
| 2020 |       98.51 |
| 2021 |       97.86 |
| 2022 |       98.2  |
| 2023 |       98.19 |
| 2024

## 8 · Tree Canopy Cover

The [USFS Tree Canopy Cover (TCC)](https://www.fs.usda.gov/ccrc/projects/forest-inventory-and-monitoring/annual-tree-canopy-cover-tcc) product provides annual, 30 m estimates of percent canopy cover — the same grid and time span as LCMS. Pairing TCC with the LCMS Change record gives a quantitative view of canopy loss and recovery: the Change band identifies *where* and *when* disturbance occurred; TCC measures *how much* canopy was lost and how quickly it has returned.

The chart below shows **mean percent tree canopy cover** averaged across the study area for each year from 1985 to 2025.

**What to look for at the H.J. Andrews:**
- **Large dip in mean TCC** in the mid 2020s in response to late-summer drought and wildfire
- **Gradual canopy recovery** through the 1990s–2010s as regenerating stands close their canopies — the structural signal that complements the "Vegetation Successional Growth" class in LCMS Change
- A consistently **high baseline TCC** 
- reflecting the old-growth and mature conifer forest that was never harvested during the 40-year record


In [8]:
TCC_ASSET = 'projects/gtac-data-publish/assets/TCC/Product_Version/2025-6'
TCC_BAND  = 'Science_Percent_Tree_Canopy_Cover'

tcc = (
    ee.ImageCollection(TCC_ASSET)
    .filter(ee.Filter.eq('study_area', study_area))
    .filterBounds(aoi)
)

print(f'TCC images in collection: {tcc.size().getInfo()}')

tcc_result = cl.summarize_and_chart(
    tcc,
    geometry=aoi,
    band_names=TCC_BAND,
    scale=30,
    area_format='Mean',
    title='Tree Canopy Cover — H.J. Andrews Experimental Forest 1985-2025',
    chart_type='line',
    date_format='YYYY',
    width=950,
    height=440,
)

tcc_result['chart'].show()

# ── Print numeric summary (first 3 and last 3 years) ─────────────────────────
tcc_col = next(
    (c for c in tcc_result['df'].columns if 'Canopy' in c or 'canopy' in c),
    tcc_result['df'].columns[0],
)
print(f'\nMean TCC (%) — column: "{tcc_col}"\n')
print(pd.concat([tcc_result['df'].head(3), tcc_result['df'].tail(3)]).to_markdown())


TCC images in collection: 41



Mean TCC (%) — column: "Science_Percent_Tree_Canopy_Cover"

|      |   Science_Percent_Tree_Canopy_Cover |
|-----:|------------------------------------:|
| 1985 |                             68.5874 |
| 1986 |                             69.0403 |
| 1987 |                             69.4138 |
| 2023 |                             73.1812 |
| 2024 |                             63.2365 |
| 2025 |                             60.176  |


## 9 · Next Steps

### What LCMS has shown us

In five cells of analysis you have:
- Mapped the current land cover, land use, and change signature of a 64 km² Pacific Northwest forest
- Identified every pixel where a Tree Removal or Wildfire occurred during the 40-year LCMS record
- Quantified how the balance of Trees, Shrubs, and other cover classes has shifted year by year
- Seen the severe impacts of mid 2020s wildfire and drought on both the Change record and Tree Canopy Cover
- Paired the LCMS Change record with annual Tree Canopy Cover to track canopy loss and recovery rates

---

### Go deeper with the site-specific notebooks

| Notebook | What it adds |
|----------|-------------|
| **[`hj_andrews_harvest_recovery.ipynb`](hj_andrews_harvest_recovery.ipynb)** | Compares individual experimental watershed units (WS1, WS6, etc.) with distinct harvest histories; shows how LCMS resolves the recovery arc for each 1960s–1970s clearcut separately |
| **[`macdunn_harvest_recovery.ipynb`](macdunn_harvest_recovery.ipynb)** | Applies the same framework to the OSU McDonald-Dunn Research Forest — an *actively managed* Oregon Coast Range forest with continuous commercial harvest throughout the LCMS record; a sharp contrast to the protected LTER setting at HJA |

---

### Other directions

| Idea | How |
|------|-----|
| Explore a different region | Replace the `aoi` and `study_area` in Section 2 with any location — LCMS covers all of CONUS + SE Alaska |
| Compare forest types | Eastern Oregon (Malheur, Ochoco NFs) has drier climate and fire-driven disturbance — very different Change signatures |
| Load management boundaries | Use `geeViz.getSummaryAreasLib` to load USFS forest or district boundaries: `sal.getUSFSForests(forest_name='Willamette')` |
| Batch export | Use `ee.batch.Export.image.toDrive(...)` to export annual LCMS GeoTIFFs for GIS integration |

---

### Data citation

> USFS GTAC. (2025). *Landscape Change Monitoring System v2025-11*. USDA Forest Service, Geospatial Technology and Applications Center. https://www.fs.usda.gov/lcms
>
> Google Earth Engine catalog: `projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11`

> USFS GTAC. (2025). *Tree Canopy Cover v2025-6*. USDA Forest Service, Geospatial Technology and Applications Center.
>
> Google Earth Engine catalog: `projects/gtac-data-publish/assets/TCC/Product_Version/2025-6`
